# Environmental Sound Classification with ResNet18
Can a network built for photos recognise **sounds**? Here a 5-second audio clip is turned into a **log-mel spectrogram** — an image of how energy is spread over frequency and time — and an image classifier (ResNet18) is trained on it.

The main question: **how much does ImageNet pre-training help on a tiny audio dataset?** The same network is trained three ways:
1. **From scratch** — random weights.
2. **Frozen feature extractor** — ImageNet weights stay fixed, only the final layer is trained.
3. **Full fine-tuning** — start from ImageNet weights and train everything.

**Data:** ESC-10, the 10-class subset of [ESC-50](https://github.com/karolpiczak/ESC-50) (K. J. Piczak, CC BY-NC 3.0): 400 clips, 40 per class, recorded from freesound.org.  
**Split:** the official folds — 1–3 train (240), 4 validation (80), 5 test (80). Clips cut from the same original recording always share a fold, so there is no leakage.  
**Compute:** CPU only. To keep training to a few minutes per strategy, the network sees a 2×2 average-pooled spectrogram (64 × 108) and each strategy gets 15 epochs.

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, librosa, librosa.display, torch
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
import train as T          # shared code: data download, features, model, training loop
torch.set_num_threads(4)
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False})

C:\Users\user\Desktop\projects\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Dataset overview

In [2]:
df = T.download_esc10()
print(len(df), 'clips')
pd.crosstab(df.category, df.fold, margins=True)

400 clips


fold,1,2,3,4,5,All
category,,,,,,
chainsaw,8,8,8,8,8,40
clock_tick,8,8,8,8,8,40
crackling_fire,8,8,8,8,8,40
crying_baby,8,8,8,8,8,40
dog,8,8,8,8,8,40
helicopter,8,8,8,8,8,40
rain,8,8,8,8,8,40
rooster,8,8,8,8,8,40
sea_waves,8,8,8,8,8,40


Perfectly balanced: 8 clips per class in every fold, so accuracy is a fair headline metric.

## 2. What the sounds look like
Each clip is resampled to 22,050 Hz and converted to a 128-bin log-mel spectrogram (n_fft = 1024, hop = 512 → 216 frames for 5 s), in decibels relative to the loudest point, then scaled with `(dB + 40) / 20` to roughly [-2, 2].

In [3]:
classes = sorted(df.category.unique())
fig, axes = plt.subplots(10, 2, figsize=(13, 22))
for row, c in zip(axes, classes):
    f = df[df.category == c].filename.iloc[0]
    y = T.load_audio(T.DATA / 'esc10' / 'audio' / f)
    librosa.display.waveshow(y, sr=T.SR, ax=row[0], color='#4f46e5')
    row[0].set_title(f'{T.PRETTY[c]} — waveform'); row[0].set_xlabel('')
    S = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=T.SR, n_fft=T.N_FFT, hop_length=T.HOP, n_mels=T.N_MELS), ref=np.max)
    librosa.display.specshow(S, sr=T.SR, hop_length=T.HOP, x_axis='time', y_axis='mel', ax=row[1], cmap='magma')
    row[1].set_title(f'{T.PRETTY[c]} — log-mel spectrogram'); row[1].set_xlabel('')
plt.tight_layout()

The classes have very different "shapes": the clock tick and dog bark are short vertical events, rain and sea waves are wide-spectrum noise spread over the whole clip, the rooster and crying baby show harmonic stripes, and the helicopter / chainsaw have steady low-frequency energy. That is exactly the kind of texture a CNN is good at picking up.

In [4]:
X, y, classes = T.build_dataset(df)
splits = T.split_by_fold(df, X, y)
print('input tensor:', tuple(X.shape), '| train/val/test:', [len(splits[k][1]) for k in ('train', 'val', 'test')])

input tensor: (400, 1, 128, 216) | train/val/test: [240, 80, 80]


## 3. Three training strategies
Shared set-up (see `train.py`):
- ResNet18 from `timm` (`resnet18.a1_in1k` weights); the spectrogram is average-pooled 2×2 (CPU budget) and its single channel is repeated to 3 channels to match ImageNet weights.
- AdamW with a cosine learning-rate schedule, batch size 16, 15 epochs.
- Light augmentation: random circular time shift plus one frequency mask and one time mask (SpecAugment).
- After every epoch the model is scored on the validation fold and the best epoch is kept.
- For the frozen variant the BatchNorm layers are kept in eval mode so their ImageNet statistics stay frozen too.

Learning rates: 1e-2 when only the small head is trained, 1e-3 otherwise. I first fine-tuned with 3e-4 (a common default): after 15 epochs it was still underfitting (train loss ≈ 1.1, validation 0.84, test 0.76). These timm weights were trained with a very different recipe (LAMB, BCE loss), and 1e-3 adapts them much better.

In [5]:
pd.DataFrame(T.STRATEGIES).T

,pretrained,freeze,lr,epochs
From scratch,False,False,0.001,15
Frozen features,True,True,0.01,15
Full fine-tuning,True,False,0.001,15


In [6]:
results = T.run_strategies(splits, verbose=False)
fig, ax = plt.subplots(figsize=(8, 4.5))
for name, r in results.items():
    h = pd.DataFrame(r['history']); ax.plot(h.epoch, h.val_acc, marker='o', ms=3, lw=2, label=name)
ax.set(xlabel='epoch', ylabel='validation accuracy (fold 4)', ylim=(0, 1), title='Validation accuracy per epoch')
ax.grid(alpha=.3); ax.legend();

In [7]:
table = pd.DataFrame([{'strategy': n, 'trainable params': f"{r['trainable_params']:,}",
                       'best val acc': round(r['best_val_acc'], 3), 'test acc': round(r['test_acc'], 3),
                       'train time (min)': round(r['train_minutes'], 1)} for n, r in results.items()])
table

,strategy,trainable params,best val acc,test acc,train time (min)
0,From scratch,"11,181,642",0.863,0.725,4.1
1,Frozen features,"5,130",0.788,0.750,1.4
2,Full fine-tuning,"11,181,642",0.975,0.913,3.9


## 4. The best model on the test fold
The final model is chosen by **validation** accuracy; the test fold is only used to report the result.

In [8]:
best_name = max(results, key=lambda k: results[k]['best_val_acc'])
best = results[best_name]['model']
Xte, yte = splits['test']
pred = T.predict_proba(best, Xte).argmax(1)
print('Best on validation:', best_name)
print(classification_report(yte, pred, target_names=[T.PRETTY[c] for c in classes], digits=3))
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(yte, pred, display_labels=[T.PRETTY[c] for c in classes],
                                        cmap='Blues', colorbar=False, ax=ax, xticks_rotation=45)
ax.set_title(f'Confusion matrix — {best_name} (test fold)');

Best on validation: Full fine-tuning
                precision    recall  f1-score   support

      Chainsaw      1.000     0.875     0.933         8
    Clock tick      0.889     1.000     0.941         8
Crackling fire      1.000     1.000     1.000         8
   Crying baby      1.000     1.000     1.000         8
      Dog bark      1.000     1.000     1.000         8
    Helicopter      0.667     0.500     0.571         8
          Rain      0.667     1.000     0.800         8
       Rooster      1.000     1.000     1.000         8
     Sea waves      1.000     0.875     0.933         8
      Sneezing      1.000     0.875     0.933         8

      accuracy                          0.912        80
     macro avg      0.922     0.912     0.911        80
  weighted avg      0.922     0.912     0.911        80



## 5. Using the model: `predict(path)` → top-3 classes

In [9]:
for c in ['dog', 'rain', 'chainsaw']:
    f = df[(df.fold == 5) & (df.category == c)].filename.iloc[0]
    print(f'{c:10s} ->', T.predict(best, T.DATA / 'esc10' / 'audio' / f, classes))

dog        -> [('dog', 0.9812), ('sneezing', 0.0111), ('clock_tick', 0.0045)]
rain       -> [('rain', 0.9693), ('helicopter', 0.0191), ('sea_waves', 0.0074)]
chainsaw   -> [('chainsaw', 0.9819), ('helicopter', 0.0076), ('sea_waves', 0.0052)]


## 6. Conclusions
- **Pre-training wins clearly — if it is allowed to adapt.** Full fine-tuning reached 97.5% validation and **91.3% test accuracy**, versus 72.5% from scratch and 75.0% with a frozen backbone.
- **Frozen ImageNet features transfer only partly.** Training just the 5,130-parameter head gives the best first epoch (53%) but plateaus below 80%: filters tuned for photos are not ideal for spectrograms.
- **From scratch overfits the tiny training set.** 11M parameters and 240 clips: 86% on validation but only 72.5% on test — a big gap, which also shows how noisy an 80-clip fold is.
- **Main weakness: helicopter vs rain.** Half of the helicopter test clips are predicted as rain — both are steady, wide-spectrum noise. Almost every other class is perfect, and the right answer is in the top 3 for 97.5% of test clips.

**Limitations.** 400 clips is very small, so the test fold (80 clips, 8 per class) gives a noisy estimate: one clip is 1.25 percentage points. The model only knows 10 classes, so any other sound is mapped to the closest of them. Training ran on CPU with a short schedule; longer training, mixup or 5-fold cross-validation would give more reliable numbers.

`train.py` reproduces this, exports the best model to ONNX and checks that the JavaScript log-mel implementation used by the web demo matches librosa.